In [1]:
import numpy as np
from sklearn.metrics import f1_score, classification_report
from typing import Tuple, Iterable, Dict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import Module
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import MNIST

In [12]:
BATCH_SIZE: int = 64

train = MNIST(root=".", download=True, train=True)
test = MNIST(root=".", download=True, train=False)

In [3]:
class MNISTTrainingSet(Dataset):
    def __init__(self, device: torch.device) -> None:
        self._device = device

        X_train, y_train = (
            [],
            [],
        )

        for image, label in train:
            image_tensor: torch.Tensor = torch.Tensor(np.asarray(image))
            label_tensor: torch.Tensor = torch.Tensor([label]).long()

            X_train.append(image_tensor)
            y_train.append(label_tensor)

        self._X = torch.stack(X_train).to(self._device)
        self._y = torch.stack(y_train).to(self._device)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self._X[index], self._y[index]

    def __len__(self) -> int:
        return len(self._X)


class MNISTTestingSet(Dataset):
    def __init__(self, device: torch.device) -> None:
        self._device = device

        X_test, y_test = [], []

        for image, label in test:
            image_tensor: torch.Tensor = torch.Tensor(np.asarray(image))
            label_tensor: torch.Tensor = torch.Tensor([label]).long()

            X_test.append(image_tensor)
            y_test.append(label_tensor)

            self._X = torch.stack(X_test).to(self._device)
            self._y = torch.stack(y_test).to(self._device)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self._X[index], self._y[index]

    def __len__(self) -> int:
        return len(self._X)


if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)
training_set = MNISTTrainingSet(device)
testing_set = MNISTTestingSet(device)

print(len(training_set), len(testing_set))

mps


/var/folders/q6/0q0r0xgd16d478qhsd_5hhlc0000gn/T/ipykernel_61153/2354640862.py:11: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  image_tensor: torch.Tensor = torch.Tensor(np.asarray(image))


60000 10000


In [ ]:
"""
    For reproducicibility a seed is manually chosen.
"""

generator: torch.Generator = torch.Generator()
generator.manual_seed(17)

train_loader: DataLoader = DataLoader(
    training_set, batch_size=BATCH_SIZE, generator=generator
)
test_loader: DataLoader = DataLoader(
    testing_set, batch_size=BATCH_SIZE, generator=generator
)

In [ ]:
def train_then_eval(model, epochs: int = 5, lr: float = 1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()

            predictions = model(batch_X)

            loss = criterion(predictions, batch_y.squeeze())
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

    print(f"Entraînement terminé. {len(train_losses)} itérations enregistrées.")

    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            predictions = model(batch_X)

            preds_classes = predictions.argmax(dim=1)

            all_preds.extend(preds_classes.cpu().numpy())
            all_targets.extend(batch_y.cpu().numpy())

    f1_global = f1_score(all_targets, all_preds, average="macro")
    accuracy = np.mean(np.array(all_preds) == np.array(all_targets))

    print(classification_report(all_targets, all_preds, digits=4))
    print(f"F1-score global (macro) : {f1_global:.4f}")
    print(f"Précision globale (Accuracy) : {accuracy:.4f}")

In [39]:
class MLP(Module):
    def __init__(self, network_size: Iterable[int], device: torch.device) -> None:
        super().__init__()

        self._device = device
        sub_models = [
            nn.Flatten(),
        ]
        for k in range(0, len(network_size) - 1):
            sub_models.append(nn.ReLU())
            sub_models.append(nn.BatchNorm1d(network_size[k], device=self._device))
            sub_models.append(
                nn.Linear(network_size[k], network_size[k + 1], device=self._device)
            )

        self._model = nn.Sequential(*sub_models)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return self._model(input)


model = MLP([784, 120, 80, 10], device)

train_then_eval(model)

Entraînement terminé. 2814 itérations enregistrées.
              precision    recall  f1-score   support

           0     0.9677    0.9796    0.9736       980
           1     0.9608    0.9930    0.9766      1135
           2     0.9625    0.9457    0.9541      1032
           3     0.9483    0.9446    0.9464      1010
           4     0.9751    0.9572    0.9661       982
           5     0.8989    0.9574    0.9273       892
           6     0.9711    0.9478    0.9593       958
           7     0.9401    0.9611    0.9505      1028
           8     0.9822    0.9045    0.9417       974
           9     0.9470    0.9554    0.9512      1009

    accuracy                         0.9552     10000
   macro avg     0.9554    0.9546    0.9547     10000
weighted avg     0.9558    0.9552    0.9552     10000

F1-score global (macro) : 0.9547
Précision globale (Accuracy) : 0.1004


In [43]:
class CNN(Module):
    def __init__(
        self,
        convolution_params: Dict[str, Iterable[int]],
        mlp_network_size: Iterable[int],
        device: torch.device,
    ) -> None:
        super().__init__()

        self._device = device
        self._convolution_params = convolution_params
        sub_models = []
        num_conv_layers = len(convolution_params["channels"]) - 1

        for k in range(num_conv_layers):
            sub_models.append(
                nn.Conv2d(
                    in_channels=convolution_params["channels"][k],
                    out_channels=convolution_params["channels"][k + 1],
                    kernel_size=convolution_params["kernel_size"][k],
                    stride=convolution_params["stride"][k],
                    padding=convolution_params["padding"][k],
                    dilation=convolution_params["dilation"][k],
                    device=self._device,
                )
            )
            sub_models.append(nn.ReLU())
            sub_models.append(nn.MaxPool2d(convolution_params["pooling_size"][k]))

        sub_models.append(nn.Flatten())
        sub_models.append(MLP(mlp_network_size, self._device))

        self._model = nn.Sequential(*sub_models)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        input = input.view(-1, 1, 28, 28)
        return self._model(input)


cnn_params = {
    "channels": [1, 4, 16],
    "kernel_size": [3, 3],
    "stride": [1, 1],
    "padding": [0, 0],
    "dilation": [1, 1],
    "pooling_size": [2, 2],
}

model = CNN(cnn_params, [400, 32, 10], device)

train_then_eval(model, epochs=10)

Entraînement terminé. 2814 itérations enregistrées.
              precision    recall  f1-score   support

           0     0.9798    0.9898    0.9848       980
           1     0.9834    0.9947    0.9890      1135
           2     0.9787    0.9777    0.9782      1032
           3     0.9909    0.9663    0.9784      1010
           4     0.9797    0.9827    0.9812       982
           5     0.9464    0.9899    0.9677       892
           6     0.9957    0.9624    0.9788       958
           7     0.9738    0.9776    0.9757      1028
           8     0.9732    0.9702    0.9717       974
           9     0.9810    0.9713    0.9761      1009

    accuracy                         0.9784     10000
   macro avg     0.9783    0.9783    0.9782     10000
weighted avg     0.9786    0.9784    0.9784     10000

F1-score global (macro) : 0.9782
Précision globale (Accuracy) : 0.1003
